In [1]:
import torch
import torch.nn as nn

# 설정
vocab_size = 5     # 어휘 사전 크기 (V)
embedding_dim = 3  # 투영층 차원 (M)

# 1. 동일한 가중치 행렬(W) 공유를 위해 시드 고정
torch.manual_seed(42)

# 방식 A: nn.Embedding 사용 (Lookup Table 방식)
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 방식 B: nn.Linear 사용 (행렬 곱 방식, 편향 bias=False)
linear_layer = nn.Linear(in_features=vocab_size, out_features=embedding_dim, bias=False)

# 두 레이어의 가중치 행렬 W를 동일하게 맞춤 (Embedding 가중치를 Linear에 전치하여 복사)
linear_layer.weight.data = embedding_layer.weight.data.T

# -------------------------------------------------------------
# 테스트: 1번 단어("cat")의 투영층 벡터 추출
target_word_idx = 1  # 1번 단어 인덱스

# A. nn.Embedding 방식
# 정수 인덱스 텐서를 입력으로 바로 전달
input_idx = torch.tensor([target_word_idx])
embed_output = embedding_layer(input_idx)

# B. nn.Linear 방식
# 원-핫 벡터 [0, 1, 0, 0, 0] 생성 후 행렬 곱 수행
one_hot_input = torch.zeros(1, vocab_size)
one_hot_input[0, target_word_idx] = 1.0
linear_output = linear_layer(one_hot_input)

# -------------------------------------------------------------
print("--- [결과 비교] ---")
print("nn.Embedding 출력 (Lookup) :\n", embed_output.detach().numpy())
print("nn.Linear    출력 (MatMul) :\n", linear_output.detach().numpy())
print("\n두 결과가 완벽히 동일한가?:", torch.allclose(embed_output, linear_output))

--- [결과 비교] ---
nn.Embedding 출력 (Lookup) :
 [[ 0.23033303 -1.1228564  -0.18632829]]
nn.Linear    출력 (MatMul) :
 [[ 0.23033303 -1.1228564  -0.18632829]]

두 결과가 완벽히 동일한가?: True


In [ ]:
import torch
import torch.nn as nn

# -------------------------------------------------------------
# 1. 설정 (이미지의 실제값 조건 적용)
# -------------------------------------------------------------
vocab_size = 3      # V = 3 (0: 사과, 1: 바나나, 2: 체리)
embedding_dim = 2   # M = 2 (2차원 공간)
target_idx = 1      # 선택할 단어: 1번 "바나나"

weight_matrix = torch.tensor([
    [0.5,  0.8],
    [-0.3, 0.9],
    [0.1, -0.4]
], dtype=torch.float32)

# -------------------------------------------------------------
# 2. nn.Embedding 구현 (Lookup 연산 방식)
# -------------------------------------------------------------
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 동일한 가중치를 사용하도록 가중치 수동 할당
embedding_layer.weight.data = weight_matrix.clone()

# 입력: 정수 인덱스 텐서 (LongTensor)
input_idx = torch.tensor([target_idx], dtype=torch.long)

# Lookup 실행
output_embed = embedding_layer(input_idx)

# -------------------------------------------------------------
# 3. nn.Linear 구현 (원-핫 벡터 행렬 곱 연산 방식)
# -------------------------------------------------------------
# bias=False 설정하여 순수 행렬 곱만 수행
linear_layer = nn.Linear(in_features=vocab_size, out_features=embedding_dim, bias=False)

# nn.Linear의 가중치는 (out_features, in_features) 형태이므로 전치(Transpose)하여 복사
linear_layer.weight.data = weight_matrix.T.clone()

# 입력: 1번 단어("바나나")에 대한 원-핫 벡터 생성 [0, 1, 0] (FloatTensor)
one_hot_input = torch.tensor([[0.0, 1.0, 0.0]], dtype=torch.float32)

# 행렬 곱 실행
output_linear = linear_layer(one_hot_input)

# -------------------------------------------------------------
# 4. 결과 출력 및 검증
# -------------------------------------------------------------
print("--- [출력 결과] ---")
print("1. nn.Embedding (Lookup) 결과 :", output_embed.detach().numpy())
print("2. nn.Linear    (MatMul) 결과 :", output_linear.detach().numpy())

# 두 결과가 수학적으로 일치하는지 확인
print("\n두 연산 결과가 완벽히 동일한가?:", torch.allclose(output_embed, output_linear))

--- [출력 결과] ---
1. nn.Embedding (Lookup) 결과 : [[-0.3  0.9]]
2. nn.Linear    (MatMul) 결과 : [[-0.3  0.9]]

두 연산 결과가 완벽히 동일한가?: True


In [3]:
import torch
import torch.nn as nn

class CBOWProjection(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWProjection, self).__init__()
        # 1. Lookup Table 역할을 하는 임베딩 레이어 (Projection Layer)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, context_indices):
        # context_indices 형태: (batch_size, num_context_words)
        # 예: [[0, 2, 3, 4]] -> 주변 단어 4개의 인덱스

        # 2. Lookup 연산으로 주변 단어 벡터들을 한 번에 추출
        # (batch_size, num_context_words, embedding_dim)
        context_embeds = self.embeddings(context_indices)

        # 3. 투영층 핵심 연산: 주변 단어 벡터들의 평균(Mean) 계산
        # (batch_size, embedding_dim)
        projection_vector = torch.mean(context_embeds, dim=1)

        return projection_vector

# 실행 예시
cbow_proj = CBOWProjection(vocab_size=10000, embedding_dim=300)
# 배치 크기 1, 주변 단어 4개 (인덱스: 12, 45, 99, 301)
context = torch.tensor([[12, 45, 99, 301]])

output_vector = cbow_proj(context)
print("\nCBOW 투영층 최종 평균 벡터 크기:", output_vector.shape) # [1, 300]


CBOW 투영층 최종 평균 벡터 크기: torch.Size([1, 300])


In [4]:
import torch
import torch.nn as nn

# 사전 정의
word2idx = {'the': 0, 'cat': 1, 'sat': 2, 'on': 3, 'mat': 4}
vocab_size = len(word2idx)
embedding_dim = 10  # M = 10차원 공간

# 입력 데이터 설정 ('the', 'cat', 'on', 'the') 및 정답 Target ('sat')
context_indices = [0, 1, 3, 0]
target_index = 2  # 중심 단어: 'sat'

# -------------------------------------------------------------
# 1. nn.Embedding 기반 CBOW (Lookup 방식)
# -------------------------------------------------------------
class CBOWEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWEmbeddingModel, self).__init__()
        # W 행렬 (Input-to-Hidden): Lookup Table
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # W' 행렬 (Hidden-to-Output)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        # inputs: [0, 1, 3, 0] (LongTensor)
        embeds = self.embeddings(inputs) # (4, 10)
        # 4개 주변 단어 임베딩의 평균 -> (1, 10) 투영층
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 5개 단어에 대한 Logits 생성
        out = self.linear(context_vector) # (1, 5)
        return out

cbow_embed = CBOWEmbeddingModel(vocab_size, embedding_dim)
context_idx = torch.tensor(context_indices, dtype=torch.long)
output_embed = cbow_embed(context_idx)

# -------------------------------------------------------------
# 2. nn.Linear 기반 CBOW (One-Hot MatMul 방식)
# -------------------------------------------------------------
class CBOWLinearModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWLinearModel, self).__init__()
        # W 행렬 (Input-to-Hidden): 행렬 곱
        self.linear1 = nn.Linear(vocab_size, embedding_dim, bias=False)
        # W' 행렬 (Hidden-to-Output)
        self.linear2 = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, one_hot_inputs):
        # one_hot_inputs: (4, 5) 원-핫 행렬 (FloatTensor)
        embeds = self.linear1(one_hot_inputs) # (4, 10) 행렬 곱
        # 4개 주변 단어 임베딩의 평균 -> (1, 10) 투영층
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 5개 단어에 대한 Logits 생성
        out = self.linear2(context_vector) # (1, 5)
        return out

cbow_linear = CBOWLinearModel(vocab_size, embedding_dim)

# 두 모델의 가중치(W, W')를 완벽히 동일하게 맞춰 결과 비교
cbow_linear.linear1.weight.data = cbow_embed.embeddings.weight.data.T.clone()
cbow_linear.linear2.weight.data = cbow_embed.linear.weight.data.clone()

# 입력: ['the', 'cat', 'on', 'the'] 4개 단어에 대한 원-핫 벡터 생성 (4, 5)
one_hot_context = torch.zeros(len(context_indices), vocab_size, dtype=torch.float32)
for i, idx in enumerate(context_indices):
    one_hot_context[i][idx] = 1.0

output_linear = cbow_linear(one_hot_context)

# -------------------------------------------------------------
# 3. 결과 비교 및 손실(Loss) 계산
# -------------------------------------------------------------
print("--- [CBOW Logits 출력 크기] ---")
print("1. nn.Embedding 기반 CBOW 출력 크기 :", output_embed.shape) # torch.Size([1, 5])
print("2. nn.Linear    기반 CBOW 출력 크기 :", output_linear.shape) # torch.Size([1, 5])

print("\n--- [연산 결과 동일 여부 확인] ---")
print("두 모델의 Logits 연산 결과가 일치하는가?:", torch.allclose(output_embed, output_linear))

# 손실 함수 및 정답 Target 설정
criterion = nn.CrossEntropyLoss()
target_tensor = torch.tensor([target_index], dtype=torch.long) # 'sat' (인덱스 2)

# 두 모델의 예측 점수(Logits)와 정답 Target 비교
loss_embed = criterion(output_embed, target_tensor)
loss_linear = criterion(output_linear, target_tensor)

print("\n--- [CBOW 손실(Loss) 계산 결과] ---")
print("1. nn.Embedding 기반 CBOW Loss :", loss_embed.item())
print("2. nn.Linear    기반 CBOW Loss :", loss_linear.item())
print("두 손실값이 완벽히 동일한가?   :", torch.allclose(loss_embed, loss_linear))

--- [CBOW Logits 출력 크기] ---
1. nn.Embedding 기반 CBOW 출력 크기 : torch.Size([1, 5])
2. nn.Linear    기반 CBOW 출력 크기 : torch.Size([1, 5])

--- [연산 결과 동일 여부 확인] ---
두 모델의 Logits 연산 결과가 일치하는가?: True

--- [CBOW 손실(Loss) 계산 결과] ---
1. nn.Embedding 기반 CBOW Loss : 1.8189280033111572
2. nn.Linear    기반 CBOW Loss : 1.8189280033111572
두 손실값이 완벽히 동일한가?   : True


In [ ]:
import torch
import torch.nn as nn

# 사전 정의
word2idx = {'the': 0, 'cat': 1, 'sat': 2, 'on': 3, 'mat': 4}
vocab_size = len(word2idx)
embedding_dim = 10  # M = 10차원 공간

# 데이터 설정 (중심 단어: 'sat'(2) / 주변 단어들: ['the'(0), 'cat'(1), 'on'(3), 'the'(0)])
target_index = 2
context_indices = [0, 1, 3, 0]

# -------------------------------------------------------------
# 1. nn.Embedding 기반 Skip-gram (Lookup 방식)
# -------------------------------------------------------------
class SkipGramEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramEmbeddingModel, self).__init__()
        # W 행렬 (Input-to-Hidden): Lookup Table
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # W' 행렬 (Hidden-to-Output)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_input):
        # target_input: [2] ('sat' 정수 인덱스)
        embed = self.embeddings(target_input) # (1, 10) 투영층
        # W'를 곱해 5개 단어에 대한 예측 점수(Logits) 생성
        out = self.linear(embed) # (1, 5)
        return out

sg_embed_model = SkipGramEmbeddingModel(vocab_size, embedding_dim)

# -------------------------------------------------------------
# 2. nn.Linear 기반 Skip-gram (One-Hot MatMul 방식)
# -------------------------------------------------------------
class SkipGramLinearModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramLinearModel, self).__init__()
        # W 행렬 (Input-to-Hidden): 행렬 곱
        self.linear1 = nn.Linear(vocab_size, embedding_dim, bias=False) # bias : False -> 음수값 제외
        # W' 행렬 (Hidden-to-Output)
        self.linear2 = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_one_hot):
        # target_one_hot: [[0.0, 0.0, 1.0, 0.0, 0.0]] (원-핫 벡터)
        embed = self.linear1(target_one_hot) # (1, 10) 행렬 곱
        # W'를 곱해 5개 단어에 대한 예측 점수(Logits) 생성
        out = self.linear2(embed) # (1, 5)
        return out

sg_linear_model = SkipGramLinearModel(vocab_size, embedding_dim) # 데이터 크기에 따라서 수정

# 두 모델의 가중치(W, W')를 완벽히 동일하게 맞춰 연산 일치 확인
sg_linear_model.linear1.weight.data = sg_embed_model.embeddings.weight.data.T.clone()
sg_linear_model.linear2.weight.data = sg_embed_model.linear.weight.data.clone()

# -------------------------------------------------------------
# 3. 입력 생성 및 예측 출력 계산
# -------------------------------------------------------------
# 1) Embedding용 입력: LongTensor 인덱스 [2]
target_idx_tensor = torch.tensor([target_index], dtype=torch.long)
output_embed = sg_embed_model(target_idx_tensor)

# 2) Linear용 입력: FloatTensor 원-핫 벡터 [0, 0, 1, 0, 0]
target_one_hot_tensor = torch.zeros(1, vocab_size, dtype=torch.float32)
target_one_hot_tensor[0][target_index] = 1.0
output_linear = sg_linear_model(target_one_hot_tensor)

# -------------------------------------------------------------
# 4. 손실(Loss) 계산 및 검증
# -------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
context_targets = torch.tensor(context_indices, dtype=torch.long) # 정답 주변 단어들 [0, 1, 3, 0]

# 1) Embedding 모델 손실 계산
loss_embed = 0
for context in context_targets:
    loss_embed += criterion(output_embed, context.unsqueeze(0))

# 2) Linear 모델 손실 계산
loss_linear = 0
for context in context_targets:
    loss_linear += criterion(output_linear, context.unsqueeze(0))

print("--- [Skip-gram 출력 크기] ---")
print("1. nn.Embedding Logits 출력 크기 :", output_embed.shape) # torch.Size([1, 5])
print("2. nn.Linear    Logits 출력 크기 :", output_linear.shape) # torch.Size([1, 5])
print("3. 두 모델의 Logits 예측 값이 완전히 일치하는가? :", torch.allclose(output_embed, output_linear))

print("\n--- [Logits 및 손실(Loss) 수치 일치 확인] ---")
print("1. nn.Embedding 총 손실(Loss)                     :", loss_embed.item())
print("2. nn.Linear    총 손실(Loss)                     :", loss_linear.item())
print("3. 두 모델의 손실(Loss)이 완전히 일치하는가?       :", torch.allclose(loss_embed, loss_linear))

--- [Skip-gram 출력 크기] ---
1. nn.Embedding Logits 출력 크기 : torch.Size([1, 5])
2. nn.Linear    Logits 출력 크기 : torch.Size([1, 5])
3. 두 모델의 Logits 예측 값이 완전히 일치하는가? : True

--- [Logits 및 손실(Loss) 수치 일치 확인] ---
1. nn.Embedding 총 손실(Loss)                     : 7.525390625
2. nn.Linear    총 손실(Loss)                     : 7.525390625
3. 두 모델의 손실(Loss)이 완전히 일치하는가?       : True


In [ ]:
import pandas as pd
from collections import Counter

df = pd.read_csv('data/daum_movie_review.csv')
# reviews = df['review'].dropna().tolist()
# print(type(reviews))
# print(reviews[:2])

sample = ['korea 한국 !!11', '@@ %%', '123 45 87 전화번호', 'xy zsldkfjae ** 한국 타이어']

def clean_text(text):
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', str(text))
    return text.strip()

result = [clean_text(r) for r in sample if len(clean_text(r)) > 0]
print(result)

tokens_list = [review.split() for review in result]
print(tokens_list)

MIN_COUNT = 1
word_counts = Counter([word for token in tokens_list for word in token])
print(word_counts)

vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]
print(vocab)

word2idx = { word: i for i, word in enumerate(vocab) }
print(word2idx)
idx2word = { i: word for i, word in enumerate(vocab) }
print(idx2word)

vocab_size = len(vocab)
print(vocab_size)

['korea 한국 11', '123 45 87 전화번호', 'xy zsldkfjae  한국 타이어']
[['korea', '한국', '11'], ['123', '45', '87', '전화번호'], ['xy', 'zsldkfjae', '한국', '타이어']]
Counter({'한국': 2, 'korea': 1, '11': 1, '123': 1, '45': 1, '87': 1, '전화번호': 1, 'xy': 1, 'zsldkfjae': 1, '타이어': 1})
['korea', '한국', '11', '123', '45', '87', '전화번호', 'xy', 'zsldkfjae', '타이어']
{'korea': 0, '한국': 1, '11': 2, '123': 3, '45': 4, '87': 5, '전화번호': 6, 'xy': 7, 'zsldkfjae': 8, '타이어': 9}
{0: 'korea', 1: '한국', 2: '11', 3: '123', 4: '45', 5: '87', 6: '전화번호', 7: 'xy', 8: 'zsldkfjae', 9: '타이어'}
10


In [1]:
print(1)

1


In [ ]:
### 문장 나누기 : KSS
### 형태소 분석 : konlp - okt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

import kss
from konlpy.tag import Okt

['돈 들인건 티가 나지만 보는 내내 하품만', '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.', '이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌... 그래도 다음 작품을 기대하게 만든다...', '이 정도면 볼만하다고 할 수 있음!', '재미있다', '나는 재밌게 봄', '0.5점은 줄 수 없냐?', '헐..다 죽었어....나중에 앤트맨 보다가도 깜놀...', '충격 결말', '응집력']


In [68]:
# -------------------------------------------------------------
# 1. 데이터 로드, 전처리 및 사전 구축
# -------------------------------------------------------------
df = pd.read_csv('data/daum_movie_review.csv')
reviews = df['review'].dropna().tolist()
print(reviews[:10])

def clean_text(text):
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', str(text))
    return text.strip()

# cleaned_reviews = [clean_text(r) for r in reviews if len(clean_text(r)) > 0]
# tokens_list = [review.split() for review in cleaned_reviews]

# 문장 분리
cleaned_reviews = []

for review in reviews:
    if len(clean_text(review)) > 0:
        cleaned_reviews.extend(kss.split_sentences(clean_text(review)))

print(cleaned_reviews[:10])

# 형태소 나누기
okt = Okt()
tokens_list = []
for reviews in cleaned_reviews:
    tokens_list.extend(okt.morphs(reviews))

# for idx, sent in enumerate(sentences, 1):
#     print(f"문장 {idx}: {sent}")

print(tokens_list[:10])

# 단어 사전 구축 (최소 빈도수 5 이상)
MIN_COUNT = 5
word_counts = Counter(tokens_list)
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

print(list(word2idx.keys())[:10])
print(vocab_size)

['돈 들인건 티가 나지만 보는 내내 하품만', '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.', '이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌... 그래도 다음 작품을 기대하게 만든다...', '이 정도면 볼만하다고 할 수 있음!', '재미있다', '나는 재밌게 봄', '0.5점은 줄 수 없냐?', '헐..다 죽었어....나중에 앤트맨 보다가도 깜놀...', '충격 결말', '응집력']
['돈 들인건 티가 나지만 보는 내내 하품만', '몰입할수밖에 없다 어렵게 생각할 필요없다', '내가 전투에 참여한듯 손에 땀이남', '이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌 그래도 다음 작품을 기대하게 만든다', '이 정도면 볼만하다고 할 수 있음', '재미있다', '나는 재밌게 봄', '05점은 줄 수 없냐', '헐다 죽었어나중에 앤트맨 보다가도 깜놀', '충격 결말']
['돈', '들인건', '티', '가', '나', '지만', '보는', '내내', '하품', '만']
['돈', '티', '가', '나', '지만', '보는', '내내', '하품', '만', '몰입']
5339


In [69]:
# -------------------------------------------------------------
# 2. CBOW 타겟-문맥 쌍 생성 및 Train/Val/Test 분할
# -------------------------------------------------------------
WINDOW_SIZE = 2
cbow_pairs = []

for tokens in tokens_list:
    indices = [word2idx[word] for word in tokens if word in word2idx]
    
    if len(indices) < WINDOW_SIZE * 2 + 1:
        continue
    for i in range(WINDOW_SIZE, len(indices) - WINDOW_SIZE):
        context = indices[i - WINDOW_SIZE:i] + indices[i + 1:i + WINDOW_SIZE + 1]
        target = indices[i]
        cbow_pairs.append((context, target))

print(cbow_pairs[:10])

# 80% Train, 10% Validation, 10% Test 비율 분할
train_pairs, test_pairs = train_test_split(cbow_pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(test_pairs, test_size=0.5, random_state=42)

print(f"전체 샘플 수    : {len(cbow_pairs)}")
print(f"Train 샘플 수   : {len(train_pairs)} (80%)")
print(f"Val 샘플 수     : {len(val_pairs)} (10%)")
print(f"Test 샘플 수    : {len(test_pairs)} (10%)")

# Dataset 클래스 (미리 tensor로 변환하도록 성능 최적화)
class CBOWDataset(Dataset):
    def __init__(self, pairs):
        contexts = [p[0] for p in pairs]
        targets = [p[1] for p in pairs]
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

train_loader = DataLoader(CBOWDataset(train_pairs), batch_size=512, shuffle=True)
val_loader = DataLoader(CBOWDataset(val_pairs), batch_size=512, shuffle=False)

[([2618, 1173, 56, 297], 2933), ([607, 4128, 297, 8], 296), ([562, 382, 1089, 1073], 381), ([117, 681, 26, 4263], 3), ([681, 3, 4263, 486], 26), ([820, 3, 343, 1359], 607), ([703, 297, 919, 615], 826), ([2028, 4358, 26, 113], 39), ([160, 160, 382, 889], 382), ([1837, 1127, 175, 191], 943)]
전체 샘플 수    : 4318
Train 샘플 수   : 3454 (80%)
Val 샘플 수     : 432 (10%)
Test 샘플 수    : 432 (10%)


In [70]:
# -------------------------------------------------------------
# 3. CBOW 모델 정의
# -------------------------------------------------------------
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        embeds = self.embeddings(inputs) # 임베딩.. 이 어떤거엿지
        context_vector = torch.mean(embeds, dim=1)
        out = self.linear(context_vector) # 출력층 행렬곱
        return out

EMBEDDING_DIM = 300
model = CBOWModel(vocab_size, EMBEDDING_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

CBOWModel(
  (embeddings): Embedding(5339, 300)
  (linear): Linear(in_features=300, out_features=5339, bias=False)
)

In [71]:
# -------------------------------------------------------------
# 4. Train 및 Validation 학습 (Best Model 저장)
# -------------------------------------------------------------
epochs = 1000
best_val_loss = float('inf')

print("\n--- CBOW 모델 학습 (Train & Validation) ---")
for epoch in range(epochs):
    # Train Phase
    model.train()
    train_loss = 0
    for context, target in train_loader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad() # 기울기 지우기
        output = model(context) # 모델링을 통한 예측값 계산
        loss = criterion(output, target) # 예측값 실제 값에 대한 차이(손실) 계산
        loss.backward() # 역전파
        optimizer.step() # 역전파 계산 적용
        train_loss += loss.item() # 한 batch에 대한 손실값 적용
    
    avg_train_loss = train_loss / len(train_loader) # 손실값의 평균 구하기

    # Validation Phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad(): # 검증은 역전파가 필요없음
        for context, target in val_loader:
            context, target = context.to(device), target.to(device)
            output = model(context)
            loss = criterion(output, target)
            val_loss += loss.item()
            
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == target).sum().item()
            val_total += target.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = (val_correct / val_total) * 100

    if epoch % 100 == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # 계속 반복하면서 최소 손실 구하기
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_cbow_model.pth')

# 통일된 파일명으로 메타 정보 저장
test_meta = {
    'word2idx': word2idx,
    'idx2word': idx2word,
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'test_pairs': test_pairs
}

with open('cbow_test_meta.pkl', 'wb') as f:
    pickle.dump(test_meta, f)

print("\n[저장 완료] 'best_cbow_model.pth' 및 메타데이터 저장됨.")


--- CBOW 모델 학습 (Train & Validation) ---
Epoch 01/1000 | Train Loss: 8.3039 | Val Loss: 7.9402 | Val Acc: 20.83%
Epoch 101/1000 | Train Loss: 0.4695 | Val Loss: 1.5213 | Val Acc: 67.59%
Epoch 201/1000 | Train Loss: 0.2888 | Val Loss: 1.7266 | Val Acc: 67.59%
Epoch 301/1000 | Train Loss: 0.2309 | Val Loss: 1.9586 | Val Acc: 67.13%
Epoch 401/1000 | Train Loss: 0.2018 | Val Loss: 2.1752 | Val Acc: 66.67%
Epoch 501/1000 | Train Loss: 0.1841 | Val Loss: 2.3880 | Val Acc: 66.90%
Epoch 601/1000 | Train Loss: 0.1742 | Val Loss: 2.5959 | Val Acc: 66.90%
Epoch 701/1000 | Train Loss: 0.1659 | Val Loss: 2.7917 | Val Acc: 67.13%
Epoch 801/1000 | Train Loss: 0.1593 | Val Loss: 2.9920 | Val Acc: 66.90%
Epoch 901/1000 | Train Loss: 0.1538 | Val Loss: 3.1905 | Val Acc: 66.90%

[저장 완료] 'best_cbow_model.pth' 및 메타데이터 저장됨.


In [72]:
# -------------------------------------------------------------
# 5. 저장된 모델 로드 후 Test 데이터셋 평가
# -------------------------------------------------------------
print("\n--- [Test Phase] 저장된 모델 로드 및 평가 ---")

# 1) 통일된 파일명으로 메타 정보 불러오기
with open('cbow_test_meta.pkl', 'rb') as f:
    loaded_meta = pickle.load(f)

# 2) 모델 구조 재구성 및 저장된 가중치(.pth) 로드
test_model = CBOWModel(loaded_meta['vocab_size'], loaded_meta['embedding_dim'])
test_model.load_state_dict(torch.load('best_cbow_model.pth')) # 불러온 모델 변수 저장
test_model.to(device)
test_model.eval() # 평가 모드

# 3) Test DataLoader 생성 및 평가 진행
loaded_test_loader = DataLoader(CBOWDataset(loaded_meta['test_pairs']), batch_size=512, shuffle=False)

test_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for context, target in loaded_test_loader:
        context, target = context.to(device), target.to(device)
        output = test_model(context)
        loss = criterion(output, target)
        test_loss += loss.item()
        
        preds = torch.argmax(output, dim=1)
        test_correct += (preds == target).sum().item()
        test_total += target.size(0)

avg_test_loss = test_loss / len(loaded_test_loader)
test_acc = (test_correct / test_total) * 100

print(f"Test Dataset Loss : {avg_test_loss:.4f}") # 0.01 미만이 좋음
print(f"Test Accuracy     : {test_acc:.2f}% ({test_correct}/{test_total} 개 적중)")


--- [Test Phase] 저장된 모델 로드 및 평가 ---
Test Dataset Loss : 1.7571
Test Accuracy     : 65.28% (282/432 개 적중)


In [73]:
# -------------------------------------------------------------
# 6. 저장된 모델 기준 코사인 유사도(Cosine Similarity) 평가
# -------------------------------------------------------------
def get_similar_words(word, model, meta, top_n=5):
    w2i = meta['word2idx']
    i2w = meta['idx2word']
    
    if word not in w2i:
        return f"'{word}' 단어가 어휘 사전에 없습니다."
    
    word_idx = torch.tensor([w2i[word]]).to(device)
    embed_weights = model.embeddings.weight.data # 임베딩 가중치 불러오기
    
    target_vec = embed_weights[word_idx] # 단어의 인덱스에 해당되는 가중치 지정
    
    # 코사인 유사도 계산
    norm_target = target_vec / torch.norm(target_vec, dim=1, keepdim=True)
    norm_weights = embed_weights / torch.norm(embed_weights, dim=1, keepdim=True)
    
    cosine_sim = torch.mm(norm_target, norm_weights.T).squeeze(0)
    top_indices = torch.topk(cosine_sim, top_n + 1).indices.tolist()
    
    return [(i2w[idx], round(cosine_sim[idx].item(), 4)) for idx in top_indices if i2w[idx] != word][:top_n]

print("\n--- [코사인 유사도 기반 상위 유사 단어 평가] ---")
test_words = ['영화', '최고의', '연기', '마블', '스토리']
for tw in test_words:
    print(f"[{tw}] 와 유사한 단어 top 5:", get_similar_words(tw, test_model, loaded_meta))


--- [코사인 유사도 기반 상위 유사 단어 평가] ---
[영화] 와 유사한 단어 top 5: [('착하게', 0.2087), ('깜놀', 0.2028), ('터', 0.1894), ('약간', 0.1891), ('종합', 0.1886)]
[최고의] 와 유사한 단어 top 5: '최고의' 단어가 어휘 사전에 없습니다.
[연기] 와 유사한 단어 top 5: [('좋아하지', 0.2101), ('보고나니', 0.1928), ('희', 0.1902), ('거짓말', 0.1884), ('행사', 0.1809)]
[마블] 와 유사한 단어 top 5: [('흐름', 0.2161), ('그대', 0.2018), ('괜찮은데', 0.1859), ('찍은', 0.1831), ('웃기는', 0.1807)]
[스토리] 와 유사한 단어 top 5: [('뮤직', 0.2358), ('도중', 0.1862), ('도경수', 0.1763), ('올해', 0.1666), ('존나', 0.1665)]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 로드, 전처리 및 사전 구축
# -------------------------------------------------------------
df = pd.read_csv('data/daum_movie_review.csv')
reviews = df['review'].dropna().tolist()

def clean_text(text):
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', str(text))
    return text.strip()

cleaned_reviews = [clean_text(r) for r in reviews if len(clean_text(r)) > 0]
tokens_list = [review.split() for review in cleaned_reviews]

# 단어 사전 구축 (최소 빈도수 5 이상)
MIN_COUNT = 5
word_counts = Counter([word for tokens in tokens_list for word in tokens])
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# -------------------------------------------------------------
# 2. Skip-gram (중심 단어 -> 주변 단어) 쌍 생성 및 데이터 분할
# -------------------------------------------------------------
WINDOW_SIZE = 2
skipgram_pairs = []

for tokens in tokens_list:
    indices = [word2idx[word] for word in tokens if word in word2idx]
    if len(indices) < WINDOW_SIZE * 2 + 1:
        continue
    for i in range(WINDOW_SIZE, len(indices) - WINDOW_SIZE):
        center_word = indices[i]
        context_words = indices[i - WINDOW_SIZE:i] + indices[i + 1:i + WINDOW_SIZE + 1]
        for context_word in context_words:
            skipgram_pairs.append((center_word, context_word))

# Train(80%), Val(10%), Test(10%) 분할
train_pairs, temp_pairs = train_test_split(skipgram_pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

print(f"전체 샘플 수    : {len(skipgram_pairs)}")
print(f"Train 샘플 수   : {len(train_pairs)} (80%)")
print(f"Val 샘플 수     : {len(val_pairs)} (10%)")
print(f"Test 샘플 수    : {len(test_pairs)} (10%)")

# Dataset 개선: __init__에서 미리 Tensor 변환
class SkipGramDataset(Dataset):
    def __init__(self, pairs):
        centers = [p[0] for p in pairs]
        contexts = [p[1] for p in pairs]
        self.centers = torch.tensor(centers, dtype=torch.long)
        self.contexts = torch.tensor(contexts, dtype=torch.long)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        return self.centers[idx], self.contexts[idx]

train_loader = DataLoader(SkipGramDataset(train_pairs), batch_size=512, shuffle=True)
val_loader = DataLoader(SkipGramDataset(val_pairs), batch_size=512, shuffle=False)

# -------------------------------------------------------------
# 3. Skip-gram 모델 정의
# -------------------------------------------------------------
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        out = self.linear(embeds)
        return out

EMBEDDING_DIM = 50
model = SkipGramModel(vocab_size, EMBEDDING_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# -------------------------------------------------------------
# 4. Train 및 Validation 학습 (Best Model 저장)
# -------------------------------------------------------------
epochs = 5
best_val_loss = float('inf')

print("\n--- Skip-gram 모델 학습 (Train & Validation) ---")
for epoch in range(epochs):
    # Train Phase
    model.train()
    train_loss = 0
    for center, context in train_loader:
        center, context = center.to(device), context.to(device)
        optimizer.zero_grad()
        output = model(center)
        loss = criterion(output, context)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for center, context in val_loader:
            center, context = center.to(device), context.to(device)
            output = model(center)
            loss = criterion(output, context)
            val_loss += loss.item()
            
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == context).sum().item()
            val_total += context.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = (val_correct / val_total) * 100

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_skipgram_model.pth')

test_meta = {
    'word2idx': word2idx,
    'idx2word': idx2word,
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'test_pairs': test_pairs
}
with open('skipgram_test_meta.pkl', 'wb') as f:
    pickle.dump(test_meta, f)

print("\n[저장 완료] 'best_skipgram_model.pth' 및 메타데이터 저장됨.")

# -------------------------------------------------------------
# 5. 저장된 모델 로드 후 Test 데이터셋 평가
# -------------------------------------------------------------
print("\n--- [Test Phase] 저장된 모델 로드 및 평가 ---")

with open('skipgram_test_meta.pkl', 'rb') as f:
    loaded_meta = pickle.load(f)

test_model = SkipGramModel(loaded_meta['vocab_size'], loaded_meta['embedding_dim'])
test_model.load_state_dict(torch.load('best_skipgram_model.pth'))
test_model.to(device)
test_model.eval()

loaded_test_loader = DataLoader(SkipGramDataset(loaded_meta['test_pairs']), batch_size=512, shuffle=False)

test_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for center, context in loaded_test_loader:
        center, context = center.to(device), context.to(device)
        output = test_model(center)
        loss = criterion(output, context)
        test_loss += loss.item()
        
        preds = torch.argmax(output, dim=1)
        test_correct += (preds == context).sum().item()
        test_total += context.size(0)

avg_test_loss = test_loss / len(loaded_test_loader)
test_acc = (test_correct / test_total) * 100

print(f"Test Dataset Loss : {avg_test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}% ({test_correct}/{test_total} 개 적중)")

# -------------------------------------------------------------
# 6. 저장된 모델 기준 코사인 유사도(Cosine Similarity) 평가
# -------------------------------------------------------------
def get_similar_words(word, model, meta, top_n=5):
    w2i = meta['word2idx']
    i2w = meta['idx2word']
    
    if word not in w2i:
        return f"'{word}' 단어가 어휘 사전에 없습니다."
    
    word_idx = torch.tensor([w2i[word]]).to(device)
    embed_weights = model.embeddings.weight.data
    
    target_vec = embed_weights[word_idx]
    
    norm_target = target_vec / torch.norm(target_vec, dim=1, keepdim=True)
    norm_weights = embed_weights / torch.norm(embed_weights, dim=1, keepdim=True)
    
    cosine_sim = torch.mm(norm_target, norm_weights.T).squeeze(0)
    top_indices = torch.topk(cosine_sim, top_n + 1).indices.tolist()
    
    return [(i2w[idx], round(cosine_sim[idx].item(), 4)) for idx in top_indices if i2w[idx] != word][:top_n]

print("\n--- [코사인 유사도 기반 상위 유사 단어 평가] ---")
test_words = ['영화', '최고의', '연기', '마블', '스토리']
for tw in test_words:
    print(f"[{tw}] 와 유사한 단어 top 5:", get_similar_words(tw, test_model, loaded_meta))